In [ ]:
# %% 셀 1: 데이터 로드 (텔롭 bbox 예측 모델)
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
HEATMAP_DIR = "./data/8_heatmap"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개")
print(f"   train: {len(train_samples):,}  eval: {len(eval_samples_with):,}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024


로드: 100%|██████████| 66400/66400 [00:16<00:00, 3983.32it/s]


✅ 채널: 664개
   train: 56,892  eval: 2,984


In [ ]:
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]
train_samples = [s for s in train_samples if len(s["instances"]) > 0]

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")


🔬 샘플링: 67개 채널
   train: 5,574  |  eval: 335


In [ ]:
# %% 셀 2: Dataset + DataLoader (instance-first, avg_coactive, inst_embs)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS_DL = 8
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 9
TIME_DIM = 3
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE
N_PATCHES_Y = GRID_H // PATCH_SIZE
N_PATCHES = N_PATCHES_X * N_PATCHES_Y


class FrameMaskViTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = [s for s in samples if len(s["instances"]) > 0]
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)
        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))

        n_inst = len(instances)
        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
        inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
        inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
        inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

        inst_bbox = np.stack([inst_x0, inst_y0, inst_x1, inst_y1], axis=-1).astype(np.int32)

        # ── 임베딩 ──
        channel_emb = F.normalize(self.text2emb.get(s["channel"], ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)
        inst_embs   = torch.stack([
            self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
        ])
        inst_embs = F.normalize(inst_embs, dim=-1)

        inst_diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        inst_diff_vid = inst_embs - video_emb.unsqueeze(0)
        inst_diff_stt = torch.zeros(n_inst, EMB_DIM)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs_raw = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
            ])
            stt_embs = F.normalize(stt_embs_raw, dim=-1)
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    inst_diff_stt[i] = inst_embs[i] - stt_embs[stt_active_idx[0]]
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0),
                        stt_embs[stt_active_idx[0]].unsqueeze(0),
                    ).item()
                    has_stts[i] = 1.0

        # ── 프레임별 활성 매트릭스 ──
        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (
            (inst_starts[None, :] <= times[:, None] + 0.05) &
            (inst_ends[None, :]   >  times[:, None])
        )

        # ── avg_coactive (log1p scaled, max ≈ 20 co-active) ──
        co_active_per_frame = active_matrix.sum(axis=1)
        inst_avg_coactive = np.zeros(n_inst, dtype=np.float32)
        for i in range(n_inst):
            frames_i = active_matrix[:, i]
            if frames_i.any():
                inst_avg_coactive[i] = co_active_per_frame[frames_i].mean()
        inst_avg_coactive = np.log1p(inst_avg_coactive) / np.log1p(20.0)

        # ── inst_scalars 9d ──
        inst_scalars = np.zeros((n_inst, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = inst_tlens / MAX_TEXT_LEN
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stts
        inst_scalars[:, 5] = inst_starts / max(duration, 0.1)
        inst_scalars[:, 6] = inst_ends / max(duration, 0.1)
        inst_scalars[:, 7] = (inst_ends - inst_starts) / max(duration, 0.1)
        inst_scalars[:, 8] = inst_avg_coactive

        # ── 프레임별 slot 매핑 (eval용) ──
        telop_mask    = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        slot_bbox     = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, 4), dtype=np.int16)
        slot_inst_idx = np.full((n_frames, MAX_ACTIVE_PER_FRAME), -1, dtype=np.int64)

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]
                for si, ai in enumerate(sorted_idx):
                    telop_mask[fi, si] = True
                    slot_bbox[fi, si] = [inst_x0[ai], inst_y0[ai],
                                         inst_x1[ai], inst_y1[ai]]
                    slot_inst_idx[fi, si] = ai

        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        return {
            "channel_id":    torch.tensor(channel_id, dtype=torch.long),
            "inst_diff_ch":  inst_diff_ch,
            "inst_diff_vid": inst_diff_vid,
            "inst_diff_stt": inst_diff_stt,
            "inst_embs":     inst_embs,
            "inst_scalars":  torch.from_numpy(inst_scalars),
            "inst_bbox":     torch.from_numpy(inst_bbox),
            "n_frames":      n_frames,
            "telop_mask":    torch.from_numpy(telop_mask),
            "time_feats":    torch.from_numpy(time_feats),
            "slot_bbox":     torch.from_numpy(slot_bbox.astype(np.int32)),
            "slot_inst_idx": torch.from_numpy(slot_inst_idx),
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    max_inst = max(b["inst_diff_ch"].shape[0] for b in batch)
    B = len(batch)

    channel_ids   = torch.stack([b["channel_id"] for b in batch])
    inst_diff_ch  = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_vid = torch.zeros(B, max_inst, EMB_DIM)
    inst_diff_stt = torch.zeros(B, max_inst, EMB_DIM)
    inst_embs     = torch.zeros(B, max_inst, EMB_DIM)
    inst_scalars  = torch.zeros(B, max_inst, SCALAR_DIM)
    inst_bbox     = torch.zeros(B, max_inst, 4, dtype=torch.int32)

    telop_mask    = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, dtype=torch.bool)
    time_feats    = torch.zeros(B, max_T, TIME_DIM)
    slot_bbox     = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, 4, dtype=torch.int32)
    frame_mask    = torch.zeros(B, max_T, dtype=torch.bool)
    slot_inst_idx = torch.full((B, max_T, MAX_ACTIVE_PER_FRAME), -1, dtype=torch.long)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        n_inst = b["inst_diff_ch"].shape[0]
        inst_diff_ch[i, :n_inst]  = b["inst_diff_ch"]
        inst_diff_vid[i, :n_inst] = b["inst_diff_vid"]
        inst_diff_stt[i, :n_inst] = b["inst_diff_stt"]
        inst_embs[i, :n_inst]     = b["inst_embs"]
        inst_scalars[i, :n_inst]  = b["inst_scalars"]
        inst_bbox[i, :n_inst]     = b["inst_bbox"]
        telop_mask[i, :T]    = b["telop_mask"]
        time_feats[i, :T]    = b["time_feats"]
        slot_bbox[i, :T]     = b["slot_bbox"]
        frame_mask[i, :T]    = True
        slot_inst_idx[i, :T] = b["slot_inst_idx"]

    return {
        "channel_ids":   channel_ids,
        "inst_diff_ch":  inst_diff_ch,
        "inst_diff_vid": inst_diff_vid,
        "inst_diff_stt": inst_diff_stt,
        "inst_embs":     inst_embs,
        "inst_scalars":  inst_scalars,
        "inst_bbox":     inst_bbox,
        "telop_mask":    telop_mask,
        "time_feats":    time_feats,
        "slot_bbox":     slot_bbox,
        "frame_mask":    frame_mask,
        "slot_inst_idx": slot_inst_idx,
    }


train_ds = FrameMaskViTDataset(train_samples, channel2id, text2emb)
eval_ds  = FrameMaskViTDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")
print(f"   EMB_DIM={EMB_DIM}  SCALAR_DIM={SCALAR_DIM}  MAX_ACTIVE={MAX_ACTIVE_PER_FRAME}")
print(f"   features: diff_ch + diff_vid + diff_stt + inst_embs (normalized 1024d each)")
print(f"           + scalars 9d (..., avg_coactive log1p-scaled)")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

In [ ]:
# %% 셀 3: 모델 정의 (instance-first + frame-local self-attn residual + co-active conditioning)

D_MODEL = 256
N_HEADS = 8
N_LAYERS_INST = 4
N_LAYERS_FRAME = 1
D_FF = 512
DROPOUT = 0.1
FRAME_CHUNK = 512


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, n_heads=N_HEADS,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.ch_proj = nn.Sequential(
            nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(
            nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(
            nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(
            nn.Linear(5, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.channel_emb = nn.Embedding(n_channels, d_model)

        # ── base prediction ──
        self.x_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_W))
        self.y_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_H))

        # ── frame-local self-attention residual ──
        self.slot_rank_emb = nn.Embedding(MAX_ACTIVE_PER_FRAME, d_model)
        nn.init.normal_(self.slot_rank_emb.weight, mean=0.0, std=0.02)

        # ── co-active position conditioning ──
        # 각 슬롯의 base x/y 분포를 frame self-attn 입력에 conditioning으로 추가
        self.base_pos_proj = nn.Sequential(
            nn.Linear(GRID_W + GRID_H, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))

        frame_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.frame_slot_attn = nn.TransformerEncoder(
            frame_layer, num_layers=N_LAYERS_FRAME, enable_nested_tensor=False)

        self.x_delta = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_W))
        self.y_delta = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_H))
        # zero-init refinement: 시작 시 final = base, delta는 필요할 때 학습됨
        nn.init.zeros_(self.x_delta[-1].weight)
        nn.init.zeros_(self.x_delta[-1].bias)
        nn.init.zeros_(self.y_delta[-1].weight)
        nn.init.zeros_(self.y_delta[-1].bias)
        self.res_scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars,
                telop_mask=None, slot_inst_idx=None, frame_mask=None):
        # ── 1. inst projection ──
        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([
            inst_scalars[..., 0:1],
            inst_scalars[..., 5:8],
            inst_scalars[..., 8:9],
        ], dim=-1)

        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)

        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1)
        )

        # ── 2. channel + self-attn ──
        ch_emb = self.channel_emb(channel_ids).unsqueeze(1)
        inst_tokens = inst_tokens + ch_emb

        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask

        inst_tokens = self.inst_self_attn(
            inst_tokens, src_key_padding_mask=inst_pad)

        # ── 3. base prediction ──
        base_x = self.x_head(inst_tokens)
        base_y = self.y_head(inst_tokens)

        mask_f = inst_mask.unsqueeze(-1).float()
        base_x = base_x * mask_f
        base_y = base_y * mask_f

        if telop_mask is None:
            return base_x, base_y, None, None

        # ── 4. frame-local self-attention residual ──
        B, T, K = telop_mask.shape
        D = inst_tokens.shape[-1]
        device = inst_tokens.device
        dtype = inst_tokens.dtype
        BT = B * T

        active_mask = telop_mask & frame_mask.unsqueeze(-1)

        idx_flat = slot_inst_idx.reshape(B, T * K).clamp(min=0)

        slot_tokens = inst_tokens.gather(
            1, idx_flat.unsqueeze(-1).expand(-1, -1, D)
        ).reshape(B, T, K, D)

        # base predictions per slot (used both for conditioning and final residual)
        base_x_slot = base_x.gather(
            1, idx_flat.unsqueeze(-1).expand(-1, -1, GRID_W)
        ).reshape(B, T, K, GRID_W)
        base_y_slot = base_y.gather(
            1, idx_flat.unsqueeze(-1).expand(-1, -1, GRID_H)
        ).reshape(B, T, K, GRID_H)

        # ── co-active position conditioning ──
        # detach: base 학습이 delta path로부터 영향 안 받게 분리
        base_pos_input = torch.cat([
            torch.sigmoid(base_x_slot.detach()),
            torch.sigmoid(base_y_slot.detach()),
        ], dim=-1)  # (B, T, K, GRID_W + GRID_H)
        base_pos_emb = self.base_pos_proj(base_pos_input)  # (B, T, K, D)

        rank_idx = torch.arange(K, device=device)
        slot_tokens = slot_tokens + \
                      self.slot_rank_emb(rank_idx).unsqueeze(0).unsqueeze(0) + \
                      base_pos_emb

        slot_tokens_flat = slot_tokens.reshape(BT, K, D)
        slot_pad_flat = ~active_mask.reshape(BT, K)

        # all-pad frames는 attention에서 NaN 위험 → active 1개 이상인 frame만 처리
        valid_frames = ~slot_pad_flat.all(dim=1)  # (BT,)
        valid_bt = valid_frames.nonzero(as_tuple=True)[0]

        ctx_tokens_flat = torch.zeros_like(slot_tokens_flat)

        if len(valid_bt) > 0:
            valid_tokens = slot_tokens_flat[valid_bt]
            valid_pad = slot_pad_flat[valid_bt]
            ctx_list = []
            for start in range(0, len(valid_bt), FRAME_CHUNK):
                end = min(start + FRAME_CHUNK, len(valid_bt))
                ctx_out = self.frame_slot_attn(
                    valid_tokens[start:end],
                    src_key_padding_mask=valid_pad[start:end],
                )
                ctx_list.append(ctx_out)
            ctx_tokens_flat[valid_bt] = torch.cat(ctx_list, dim=0)

        ctx_tokens = ctx_tokens_flat.reshape(B, T, K, D)

        delta_x = self.x_delta(ctx_tokens)
        delta_y = self.y_delta(ctx_tokens)

        final_x = base_x_slot + self.res_scale * delta_x
        final_y = base_y_slot + self.res_scale * delta_y

        active_f = active_mask.unsqueeze(-1).float()
        final_x = final_x * active_f
        final_y = final_y * active_f

        return base_x, base_y, final_x, final_y


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   inst-first: ch/vid/stt/len proj → 256d → self-attn {N_LAYERS_INST}L → base x/y")
print(f"   co-active conditioning: base_pos_emb = MLP(sigmoid base_x/y, 160→256d, detached)")
print(f"   frame residual: frame-local self-attn {N_LAYERS_FRAME}L (slot_token + rank_emb + base_pos_emb)")
print(f"   final = base + res_scale * delta  (delta zero-init, res_scale init=0.1)")
print(f"   output: x[{GRID_W}] + y[{GRID_H}] per (frame, slot)")

In [ ]:
# %% 셀 4: 학습 (inst focal + hungarian-matched frame loss + center/height + diagnostics)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.optimize import linear_sum_assignment

EPOCHS = 50
LR = 1e-4
FOCAL_ALPHA_X = 2.7 / (1 + 2.7)
FOCAL_ALPHA_Y = 25.6 / (1 + 25.6)
FOCAL_GAMMA = 2.0
HUNG_WEIGHT = 0.5
HUNG_LAMBDA_TEXT = 0.3
HUNG_LAMBDA_TIME = 0.3
HUNG_MAX_FRAMES = 200
CENTER_HEIGHT_WEIGHT = 0.2
SAVE_DIR = "./model/8_layout_slot_xy"
os.makedirs(SAVE_DIR, exist_ok=True)

_x_range = torch.arange(GRID_W, device=device).unsqueeze(0)
_y_range = torch.arange(GRID_H, device=device).unsqueeze(0)
_x_range_f = torch.arange(GRID_W, device=device).float()
_y_range_f = torch.arange(GRID_H, device=device).float()


def bbox_to_gt_xy_inst(inst_bbox, inst_mask):
    bbox = inst_bbox[inst_mask].long()
    x0, y0, x1, y1 = bbox[:, 0], bbox[:, 1], bbox[:, 2], bbox[:, 3]
    gt_x = ((_x_range >= x0.unsqueeze(1)) & (_x_range < x1.unsqueeze(1))).float()
    gt_y = ((_y_range >= y0.unsqueeze(1)) & (_y_range < y1.unsqueeze(1))).float()
    return gt_x, gt_y


def focal_loss_1d(logits, targets, alpha, gamma=FOCAL_GAMMA):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs = torch.sigmoid(logits)
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    focal_weight = alpha_t * (1 - p_t) ** gamma
    return (focal_weight * bce).mean()


def inst_xy_loss(x_logits, y_logits, inst_bbox, inst_mask):
    if not inst_mask.any():
        return x_logits.sum() * 0.0
    active_x = x_logits[inst_mask]
    active_y = y_logits[inst_mask]
    gt_x, gt_y = bbox_to_gt_xy_inst(inst_bbox, inst_mask)
    return focal_loss_1d(active_x, gt_x, alpha=FOCAL_ALPHA_X) + \
           focal_loss_1d(active_y, gt_y, alpha=FOCAL_ALPHA_Y)


def xy_center_height_loss(final_x, final_y, slot_bbox, active_mask):
    """
    Predicted distribution centroid이 GT center에, 총 mass가 GT width/height에 맞도록 강제.
    bbox가 [x0, x1) 반열린 구간이라 포함 index 중심은 (x0+x1-1)/2.
    """
    if not active_mask.any():
        return final_x.sum() * 0.0

    xl = final_x[active_mask]
    yl = final_y[active_mask]
    bbox = slot_bbox[active_mask].long()
    x0 = bbox[:, 0].float()
    y0 = bbox[:, 1].float()
    x1 = bbox[:, 2].float()
    y1 = bbox[:, 3].float()

    gt_cx = (x0 + x1 - 1) / 2  # 포함 index 중심 (반열린 구간)
    gt_cy = (y0 + y1 - 1) / 2
    gt_w  = (x1 - x0).clamp(min=1.0)
    gt_h  = (y1 - y0).clamp(min=1.0)

    px = torch.sigmoid(xl)
    py = torch.sigmoid(yl)
    mass_x = px.sum(dim=-1).clamp(min=1e-6)
    mass_y = py.sum(dim=-1).clamp(min=1e-6)

    pred_cx = (px * _x_range_f).sum(dim=-1) / mass_x
    pred_cy = (py * _y_range_f).sum(dim=-1) / mass_y

    center_l = (pred_cx - gt_cx).abs() / GRID_W + (pred_cy - gt_cy).abs() / GRID_H
    width_l  = (mass_x - gt_w).abs() / GRID_W
    height_l = (mass_y - gt_h).abs() / GRID_H

    return (center_l + width_l + height_l).mean()


def hungarian_frame_loss(final_x, final_y, slot_bbox, active_mask,
                         inst_embs, slot_inst_idx, inst_scalars,
                         deterministic=False):
    """
    Per-frame Hungarian-matched focal loss with text + time aware cost.
    Returns (loss, stats) where stats contains diagnostic counters.
    deterministic=True: eval에서 frame sampling을 linspace로 고정해서 eval_loss 안정화.
    """
    B, T, K = active_mask.shape
    BT = B * T
    dev = final_x.device

    n_active_per_frame = active_mask.reshape(BT, K).sum(dim=1)
    valid_flat = (n_active_per_frame >= 1)
    valid_idx = valid_flat.nonzero(as_tuple=True)[0]

    stats = {
        'n_pairs': 0,
        'n_offdiag': 0,
        'offdiag_text_sim_sum': 0.0,
        'offdiag_time_dist_sum': 0.0,
    }

    if len(valid_idx) == 0:
        return final_x.sum() * 0.0, stats

    if len(valid_idx) > HUNG_MAX_FRAMES:
        if deterministic:
            # 균등 sampling (eval용, 결정적)
            pick = torch.linspace(0, len(valid_idx) - 1, HUNG_MAX_FRAMES, device=dev).long()
            valid_idx = valid_idx[pick]
        else:
            # 랜덤 sampling (train용)
            sample = torch.randperm(len(valid_idx), device=dev)[:HUNG_MAX_FRAMES]
            valid_idx = valid_idx[sample]

    final_x_flat = final_x.reshape(BT, K, GRID_W)
    final_y_flat = final_y.reshape(BT, K, GRID_H)
    slot_bbox_flat = slot_bbox.reshape(BT, K, 4)
    active_mask_flat = active_mask.reshape(BT, K)
    slot_inst_idx_flat = slot_inst_idx.reshape(BT, K)

    batch_idx_for_bt = torch.arange(B, device=dev).unsqueeze(1).expand(B, T).reshape(BT)

    x_range_full = torch.arange(GRID_W, device=dev)
    y_range_full = torch.arange(GRID_H, device=dev)

    total_loss = final_x.sum() * 0.0
    n_processed = 0

    for fi in valid_idx.tolist():
        bi = batch_idx_for_bt[fi].item()
        am = active_mask_flat[fi]
        active_slots = am.nonzero(as_tuple=True)[0]
        n_a = len(active_slots)

        inst_idx_at_slots = slot_inst_idx_flat[fi, active_slots]
        embs = inst_embs[bi, inst_idx_at_slots].float()
        starts = inst_scalars[bi, inst_idx_at_slots, 5].float()

        xl = final_x_flat[fi, active_slots]
        yl = final_y_flat[fi, active_slots]
        bb = slot_bbox_flat[fi, active_slots].long()

        gt_x = ((x_range_full.unsqueeze(0) >= bb[:, 0:1]) &
                (x_range_full.unsqueeze(0) <  bb[:, 2:3])).float()
        gt_y = ((y_range_full.unsqueeze(0) >= bb[:, 1:2]) &
                (y_range_full.unsqueeze(0) <  bb[:, 3:4])).float()

        if n_a == 1:
            loss_x = focal_loss_1d(xl, gt_x, alpha=FOCAL_ALPHA_X)
            loss_y = focal_loss_1d(yl, gt_y, alpha=FOCAL_ALPHA_Y)
            total_loss = total_loss + (loss_x + loss_y)
            n_processed += 1
            stats['n_pairs'] += 1
            continue

        # ── Pairwise bbox cost ──
        xl_d = xl.detach().float()
        yl_d = yl.detach().float()
        gt_x_d = gt_x.detach()
        gt_y_d = gt_y.detach()

        bce_x = F.binary_cross_entropy_with_logits(
            xl_d.unsqueeze(1).expand(-1, n_a, -1),
            gt_x_d.unsqueeze(0).expand(n_a, -1, -1),
            reduction="none"
        ).mean(dim=-1)
        bce_y = F.binary_cross_entropy_with_logits(
            yl_d.unsqueeze(1).expand(-1, n_a, -1),
            gt_y_d.unsqueeze(0).expand(n_a, -1, -1),
            reduction="none"
        ).mean(dim=-1)
        bbox_cost = bce_x + bce_y

        # ── Identity priors ──
        text_sim = embs @ embs.T
        text_dissim = 1.0 - text_sim
        time_dist = (starts.unsqueeze(1) - starts.unsqueeze(0)).abs()

        bbox_scale = bbox_cost.median().detach() + 1e-6
        cost = bbox_cost \
             + HUNG_LAMBDA_TEXT * text_dissim * bbox_scale \
             + HUNG_LAMBDA_TIME * time_dist * bbox_scale

        cost_np = cost.float().detach().cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(cost_np)

        # ── Diagnostics: count off-diagonal matchings ──
        text_sim_cpu = text_sim.detach().cpu()
        time_dist_cpu = time_dist.detach().cpu()
        for r, c in zip(row_ind.tolist(), col_ind.tolist()):
            stats['n_pairs'] += 1
            if r != c:
                stats['n_offdiag'] += 1
                stats['offdiag_text_sim_sum'] += float(text_sim_cpu[r, c])
                stats['offdiag_time_dist_sum'] += float(time_dist_cpu[r, c])

        row_t = torch.tensor(row_ind, device=dev, dtype=torch.long)
        col_t = torch.tensor(col_ind, device=dev, dtype=torch.long)

        matched_xl = xl[row_t]
        matched_yl = yl[row_t]
        matched_gt_x = gt_x[col_t]
        matched_gt_y = gt_y[col_t]

        loss_x = focal_loss_1d(matched_xl, matched_gt_x, alpha=FOCAL_ALPHA_X)
        loss_y = focal_loss_1d(matched_yl, matched_gt_y, alpha=FOCAL_ALPHA_Y)

        total_loss = total_loss + (loss_x + loss_y)
        n_processed += 1

    if n_processed == 0:
        return final_x.sum() * 0.0, stats

    return total_loss / n_processed, stats


def combined_loss(base_x, base_y, final_x, final_y,
                  inst_bbox, inst_mask, slot_bbox, active_mask,
                  inst_embs, slot_inst_idx, inst_scalars,
                  deterministic=False):
    loss_base = inst_xy_loss(base_x, base_y, inst_bbox, inst_mask)
    loss_hung, hung_stats = hungarian_frame_loss(
        final_x, final_y, slot_bbox, active_mask,
        inst_embs, slot_inst_idx, inst_scalars,
        deterministic=deterministic,
    )
    loss_ch = xy_center_height_loss(base_x, base_y, inst_bbox, inst_mask)  # base에 걸어 Hungarian과 충돌 방지
    total = loss_base + HUNG_WEIGHT * loss_hung + CENTER_HEIGHT_WEIGHT * loss_ch
    stats = {
        'loss_base': float(loss_base.detach()),
        'loss_hung': float(loss_hung.detach()),
        'loss_ch':   float(loss_ch.detach()),
        **hung_stats,
    }
    return total, stats


@torch.no_grad()
def compute_slot_metrics(final_x, final_y, telop_mask, slot_bbox, frame_mask):
    active_mask = telop_mask & frame_mask.unsqueeze(-1)

    if not active_mask.any():
        return None

    active_x = torch.sigmoid(final_x[active_mask])
    active_y = torch.sigmoid(final_y[active_mask])

    bbox = slot_bbox[active_mask].long()
    x0, y0, x1, y1 = bbox[:, 0], bbox[:, 1], bbox[:, 2], bbox[:, 3]
    gt_x_bool = ((_x_range >= x0.unsqueeze(1)) & (_x_range < x1.unsqueeze(1)))
    gt_y_bool = ((_y_range >= y0.unsqueeze(1)) & (_y_range < y1.unsqueeze(1)))

    th_list = [i / 10 for i in range(1, 10)]

    best_xf1 = 0.0
    for th in th_list:
        xb = active_x > th
        tp = (xb & gt_x_bool).sum().item()
        fp = (xb & ~gt_x_bool).sum().item()
        fn = (~xb & gt_x_bool).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if f1 > best_xf1:
            best_xf1 = f1

    best_yf1 = 0.0
    for th in th_list:
        yb = active_y > th
        tp = (yb & gt_y_bool).sum().item()
        fp = (yb & ~gt_y_bool).sum().item()
        fn = (~yb & gt_y_bool).sum().item()
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        if f1 > best_yf1:
            best_yf1 = f1

    gt_x_sum = gt_x_bool.sum(dim=-1).float()
    gt_y_sum = gt_y_bool.sum(dim=-1).float()
    gt_area = gt_x_sum * gt_y_sum
    gt_total = gt_area.sum().item()

    best_f1, best_p, best_r = 0.0, 0.0, 0.0
    best_xth, best_yth = 0.5, 0.5

    coarse = [i / 10 for i in range(1, 10)]
    for thx in coarse:
        x_bin = active_x > thx
        x_pred_sum = x_bin.sum(dim=-1).float()
        x_match = (x_bin & gt_x_bool).sum(dim=-1).float()
        for thy in coarse:
            y_bin = active_y > thy
            y_pred_sum = y_bin.sum(dim=-1).float()
            y_match = (y_bin & gt_y_bool).sum(dim=-1).float()
            tp = (x_match * y_match).sum().item()
            pred_area = (x_pred_sum * y_pred_sum).sum().item()
            fp = pred_area - tp
            fn = gt_total - tp
            p = tp / (tp + fp + 1e-8)
            r = tp / (tp + fn + 1e-8)
            f1 = 2 * p * r / (p + r + 1e-8)
            if f1 > best_f1:
                best_f1, best_p, best_r = f1, p, r
                best_xth, best_yth = thx, thy

    fine_x = [best_xth + d / 100 for d in range(-9, 10)]
    fine_y = [best_yth + d / 100 for d in range(-9, 10)]
    fine_x = [t for t in fine_x if 0.01 <= t <= 0.99]
    fine_y = [t for t in fine_y if 0.01 <= t <= 0.99]

    for thx in fine_x:
        x_bin = active_x > thx
        x_pred_sum = x_bin.sum(dim=-1).float()
        x_match = (x_bin & gt_x_bool).sum(dim=-1).float()
        for thy in fine_y:
            y_bin = active_y > thy
            y_pred_sum = y_bin.sum(dim=-1).float()
            y_match = (y_bin & gt_y_bool).sum(dim=-1).float()
            tp = (x_match * y_match).sum().item()
            pred_area = (x_pred_sum * y_pred_sum).sum().item()
            fp = pred_area - tp
            fn = gt_total - tp
            p = tp / (tp + fp + 1e-8)
            r = tp / (tp + fn + 1e-8)
            f1 = 2 * p * r / (p + r + 1e-8)
            if f1 > best_f1:
                best_f1, best_p, best_r = f1, p, r
                best_xth, best_yth = thx, thy

    result = {
        "P": best_p, "R": best_r, "F1": best_f1,
        "bestF1": best_f1,
        "bestThX": best_xth, "bestThY": best_yth,
        "xF1": best_xf1,
        "yF1": best_yf1,
    }

    B, T, K = active_mask.shape
    x_prob = torch.sigmoid(final_x)
    y_prob = torch.sigmoid(final_y)

    has_frame = active_mask.any(dim=2)
    b_idx, t_idx = has_frame.nonzero(as_tuple=True)
    n_total = len(b_idx)
    if n_total > 500:
        sample_idx = torch.randperm(n_total, device=final_x.device)[:500]
        b_idx = b_idx[sample_idx]
        t_idx = t_idx[sample_idx]

    hung_tp, hung_fp, hung_fn = 0, 0, 0
    for bi, ti in zip(b_idx.tolist(), t_idx.tolist()):
        am = active_mask[bi, ti]
        active_slots = am.nonzero(as_tuple=True)[0]
        n_active = len(active_slots)
        if n_active == 0:
            continue

        xp = x_prob[bi, ti, active_slots]
        yp = y_prob[bi, ti, active_slots]
        bb = slot_bbox[bi, ti, active_slots].long()

        pred_masks = (yp > best_yth).unsqueeze(-1) & (xp > best_xth).unsqueeze(-2)
        gt_masks = torch.zeros(n_active, GRID_H, GRID_W, device=final_x.device, dtype=torch.bool)
        for j in range(n_active):
            gt_masks[j, bb[j, 1]:bb[j, 3], bb[j, 0]:bb[j, 2]] = True

        inter = (pred_masks.unsqueeze(1) & gt_masks.unsqueeze(0)).sum(dim=(-1, -2)).float()
        union = (pred_masks.unsqueeze(1) | gt_masks.unsqueeze(0)).sum(dim=(-1, -2)).float()
        iou_matrix = inter / (union + 1e-8)

        cost = (1 - iou_matrix).cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(cost)

        for ri, ci in zip(row_ind, col_ind):
            pm = pred_masks[ri]
            gm = gt_masks[ci]
            hung_tp += (pm & gm).sum().item()
            hung_fp += (pm & ~gm).sum().item()
            hung_fn += (~pm & gm).sum().item()

    hung_p = hung_tp / (hung_tp + hung_fp + 1e-8)
    hung_r = hung_tp / (hung_tp + hung_fn + 1e-8)
    result["hungF1"] = 2 * hung_p * hung_r / (hung_p + hung_r + 1e-8)

    b_idx2, t_idx2 = has_frame.nonzero(as_tuple=True)
    n_total2 = len(b_idx2)
    if n_total2 > 2000:
        sample_idx = torch.randperm(n_total2, device=final_x.device)[:2000]
        b_idx2 = b_idx2[sample_idx]
        t_idx2 = t_idx2[sample_idx]

    merged_preds, merged_gts = [], []
    for bi, ti in zip(b_idx2.tolist(), t_idx2.tolist()):
        am = active_mask[bi, ti]
        bb = slot_bbox[bi, ti]

        xp = x_prob[bi, ti, am]
        yp = y_prob[bi, ti, am]
        pred_x_bin = xp > best_xth
        pred_y_bin = yp > best_yth
        slot_2d = pred_y_bin.unsqueeze(-1) & pred_x_bin.unsqueeze(-2)
        pred_merged = slot_2d.any(dim=0)

        gt_boxes = bb[am].long()
        gt_merged = torch.zeros(GRID_H, GRID_W, device=final_x.device, dtype=torch.bool)
        for box in gt_boxes:
            gt_merged[box[1]:box[3], box[0]:box[2]] = True

        merged_preds.append(pred_merged)
        merged_gts.append(gt_merged)

    mp = torch.stack(merged_preds).reshape(-1)
    mg = torch.stack(merged_gts).reshape(-1)

    tp = (mp & mg).sum().item()
    fp = (mp & ~mg).sum().item()
    fn = (~mp & mg).sum().item()
    p_m = tp / (tp + fp + 1e-8)
    r_m = tp / (tp + fn + 1e-8)
    result["mBestF1"] = 2 * p_m * r_m / (p_m + r_m + 1e-8)

    return result


def _empty_diag():
    return {
        'loss_base_sum': 0.0, 'loss_hung_sum': 0.0, 'loss_ch_sum': 0.0,
        'n_batches': 0,
        'n_pairs': 0, 'n_offdiag': 0,
        'offdiag_text_sim_sum': 0.0, 'offdiag_time_dist_sum': 0.0,
    }


def _accum_diag(d, stats):
    d['loss_base_sum'] += stats['loss_base']
    d['loss_hung_sum'] += stats['loss_hung']
    d['loss_ch_sum']   += stats['loss_ch']
    d['n_batches']     += 1
    d['n_pairs']       += stats['n_pairs']
    d['n_offdiag']     += stats['n_offdiag']
    d['offdiag_text_sim_sum']  += stats['offdiag_text_sim_sum']
    d['offdiag_time_dist_sum'] += stats['offdiag_time_dist_sum']


def _summarize_diag(d):
    nb = max(d['n_batches'], 1)
    np_ = max(d['n_pairs'], 1)
    no = max(d['n_offdiag'], 1)
    return {
        'loss_base': d['loss_base_sum'] / nb,
        'loss_hung': d['loss_hung_sum'] / nb,
        'loss_ch':   d['loss_ch_sum']   / nb,
        'offdiag_rate': d['n_offdiag'] / np_,
        'offdiag_text_sim_mean': d['offdiag_text_sim_sum'] / no if d['n_offdiag'] > 0 else 0.0,
        'offdiag_time_dist_mean': d['offdiag_time_dist_sum'] / no if d['n_offdiag'] > 0 else 0.0,
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0.0, 0
    train_diag = _empty_diag()

    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_ids   = batch["channel_ids"].to(device)
        inst_diff_ch  = batch["inst_diff_ch"].to(device)
        inst_diff_vid = batch["inst_diff_vid"].to(device)
        inst_diff_stt = batch["inst_diff_stt"].to(device)
        inst_embs     = batch["inst_embs"].to(device)
        inst_scalars  = batch["inst_scalars"].to(device)
        inst_bbox     = batch["inst_bbox"].to(device)
        telop_mask    = batch["telop_mask"].to(device)
        slot_bbox     = batch["slot_bbox"].to(device)
        frame_mask    = batch["frame_mask"].to(device)
        slot_inst_idx = batch["slot_inst_idx"].to(device)

        inst_mask = (inst_scalars[..., 0] != 0)
        active_mask = telop_mask & frame_mask.unsqueeze(-1)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            base_x, base_y, final_x, final_y = model(
                channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars,
                telop_mask, slot_inst_idx, frame_mask,
            )
            loss, stats = combined_loss(
                base_x, base_y, final_x, final_y,
                inst_bbox, inst_mask, slot_bbox, active_mask,
                inst_embs, slot_inst_idx, inst_scalars,
                deterministic=False,
            )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1
        _accum_diag(train_diag, stats)

    model.eval()
    eval_loss_sum, eval_batches = 0.0, 0
    eval_diag = _empty_diag()
    all_metrics = []

    with torch.no_grad():
        for batch in eval_loader:
            channel_ids   = batch["channel_ids"].to(device)
            inst_diff_ch  = batch["inst_diff_ch"].to(device)
            inst_diff_vid = batch["inst_diff_vid"].to(device)
            inst_diff_stt = batch["inst_diff_stt"].to(device)
            inst_embs     = batch["inst_embs"].to(device)
            inst_scalars  = batch["inst_scalars"].to(device)
            inst_bbox     = batch["inst_bbox"].to(device)
            telop_mask    = batch["telop_mask"].to(device)
            slot_bbox     = batch["slot_bbox"].to(device)
            frame_mask    = batch["frame_mask"].to(device)
            slot_inst_idx = batch["slot_inst_idx"].to(device)

            inst_mask = (inst_scalars[..., 0] != 0)
            active_mask = telop_mask & frame_mask.unsqueeze(-1)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                base_x, base_y, final_x, final_y = model(
                    channel_ids, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars,
                    telop_mask, slot_inst_idx, frame_mask,
                )
                loss, stats = combined_loss(
                    base_x, base_y, final_x, final_y,
                    inst_bbox, inst_mask, slot_bbox, active_mask,
                    inst_embs, slot_inst_idx, inst_scalars,
                    deterministic=True,
                )

            eval_loss_sum += loss.item()
            eval_batches += 1
            _accum_diag(eval_diag, stats)

            m = compute_slot_metrics(
                final_x, final_y, telop_mask, slot_bbox, frame_mask,
            )
            if m is not None:
                all_metrics.append(m)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss  = eval_loss_sum  / max(eval_batches, 1)
    lr_now = optimizer.param_groups[0]["lr"]

    if all_metrics:
        avg_m = {k: np.mean([m[k] for m in all_metrics])
                 for k in all_metrics[0] if "Th" not in k}
        avg_thx = np.mean([m["bestThX"] for m in all_metrics])
        avg_thy = np.mean([m["bestThY"] for m in all_metrics])
    else:
        avg_m = {"P": 0, "R": 0, "F1": 0, "bestF1": 0, "hungF1": 0,
                 "mBestF1": 0, "xF1": 0, "yF1": 0}
        avg_thx, avg_thy = 0.5, 0.5

    tr_d = _summarize_diag(train_diag)
    ev_d = _summarize_diag(eval_diag)

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m.get('P',0):.3f}  R={avg_m.get('R',0):.3f}  "
        f"F1={avg_m.get('F1',0):.3f}  best={avg_m.get('bestF1',0):.3f}  "
        f"th=({avg_thx:.2f},{avg_thy:.2f})  "
        f"hF1={avg_m.get('hungF1',0):.3f}  "
        f"mF1={avg_m.get('mBestF1',0):.3f}  "
        f"xF1={avg_m.get('xF1',0):.3f}  yF1={avg_m.get('yF1',0):.3f}  "
        f"lr={lr_now:.2e}"
    )
    print(
        f"   train: lb={tr_d['loss_base']:.3f} lh={tr_d['loss_hung']:.3f} lc={tr_d['loss_ch']:.3f} "
        f"| offdiag={tr_d['offdiag_rate']:.3f} ts={tr_d['offdiag_text_sim_mean']:.3f} td={tr_d['offdiag_time_dist_mean']:.3f}"
    )
    print(
        f"   eval:  lb={ev_d['loss_base']:.3f} lh={ev_d['loss_hung']:.3f} lc={ev_d['loss_ch']:.3f} "
        f"| offdiag={ev_d['offdiag_rate']:.3f} ts={ev_d['offdiag_text_sim_mean']:.3f} td={ev_d['offdiag_time_dist_mean']:.3f}"
    )

    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "train_diag": tr_d,
        "eval_diag":  ev_d,
        "channel2id": channel2id,
    }
    torch.save(ckpt, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save(ckpt, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")